
# Lightweight Input-Adaptive Multimodal Framework for Robust Visual–Tactile Braille Recognition

This notebook is a **full executable reference implementation** of the proposed workflow in the supplied manuscript:

**URC-APG → BrailleMamba-Lite → TactileShuffle-Lite → BCAN → BRNEN-CG → Accessible Note Layer → Explainability**

The implementation follows the manuscript's main design choices: leakage-aware group splitting, three preprocessing routes, quality-conditioned routing, shared/private multimodal representations, prototype-anchored alignment, reliability-aware fusion, missing-modality training, single-cell recognition, visual sequence decoding, calibrated selective review, deterministic note construction, Grad-CAM, integrated gradients, degradation evaluation, ablation utilities, and reproducibility logging.

> **Important:** the manuscript contains provisional/planning hardware and numerical targets. This notebook **does not hard-code those target results as measured results**. All reported metrics are computed from the data and model actually run.

The notebook runs end-to-end in `DEMO_MODE=True` using reproducible synthetic Braille/tactile data. Set `DEMO_MODE=False` and configure the dataset paths when the real datasets and label metadata are available.


In [ ]:

# 1. Installation / imports
# Purpose: install/import the packages needed by the complete pipeline.

import os, sys, math, time, json, random, pickle, warnings, copy, re
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Uncomment in a fresh Colab runtime if required:
# !pip -q install torch torchvision scikit-learn opencv-python scipy seaborn pillow

import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    roc_auc_score, average_precision_score, brier_score_loss
)
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import wilcoxon

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


In [ ]:

# 2. Reproducible configuration
# Purpose: centralize dataset paths, model sizes, training settings and risk-control parameters.

@dataclass
class Config:
    seed: int = 42
    demo_mode: bool = True

    # Real dataset paths from the manuscript
    image_root: str = "/content/Multi_Braille_Characters"
    tactile_root: str = "/content/Tactile_Braille_Letters_V2"
    label_map_csv: str = "/content/braille_label_map.csv"

    image_size: int = 96
    tactile_len: int = 256
    tactile_channels: int = 12
    embed_dim: int = 128
    token_dim: int = 64
    visual_blocks: int = 2
    tactile_blocks: int = 2
    dropout: float = 0.10

    batch_size: int = 32
    epochs: int = 12
    lr: float = 2e-3
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    patience: int = 4

    # URC-APG
    quality_dim: int = 4
    route_costs: tuple = (1.0, 1.45, 2.0)
    utility_hidden: int = 64
    cost_weights: tuple = (0.00, 0.02, 0.05, 0.10, 0.20)
    accuracy_tolerance: float = 0.01

    # BCAN / review
    temperature: float = 0.07
    review_hidden: int = 64
    review_thresholds: tuple = tuple(np.round(np.linspace(0.50, 0.95, 10), 2))
    review_error_tolerance: float = 0.02

    # Evaluation
    repetitions: int = 1
    pairing_seeds: tuple = (7, 17)
    degradation_levels: tuple = (0, 1, 2)

CFG = Config()
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(CFG.seed)
print(asdict(CFG))


In [ ]:

# 3. Canonical Grade-1 Braille utilities
# Purpose: provide six-dot canonical cell codes and deterministic note conversion.

# Dot order: 1,2,3,4,5,6
LETTER_DOTS = {
    "a": (1,), "b": (1,2), "c": (1,4), "d": (1,4,5), "e": (1,5),
    "f": (1,2,4), "g": (1,2,4,5), "h": (1,2,5), "i": (2,4), "j": (2,4,5),
    "k": (1,3), "l": (1,2,3), "m": (1,3,4), "n": (1,3,4,5), "o": (1,3,5),
    "p": (1,2,3,4), "q": (1,2,3,4,5), "r": (1,2,3,5), "s": (2,3,4),
    "t": (2,3,4,5), "u": (1,3,6), "v": (1,2,3,6), "w": (2,4,5,6),
    "x": (1,3,4,6), "y": (1,3,4,5,6), "z": (1,3,5,6),
}
PUNCT_DOTS = {
    ",": (2,), ";": (2,3), ":": (2,5), ".": (2,5,6), "?": (2,3,6),
    "!": (2,3,5), "'": (3,), "-": (3,6), "(": (2,3,5,6), ")": (2,3,5,6),
}
CHAR_TO_CODE = {c: tuple(int(i in dots) for i in range(1,7)) for c,dots in {**LETTER_DOTS, **PUNCT_DOTS}.items()}
CHAR_TO_CODE[" "] = (0,0,0,0,0,0)

# Grade-1 indicators used by the note layer.
CAPITAL_CODE = tuple(int(i in (4,6)) for i in range(1,7))
NUMBER_CODE = tuple(int(i in (3,4,5,6)) for i in range(1,7))

def code_to_int(code):
    return int("".join(map(str, code)), 2)

ALL_CODES = sorted(set(CHAR_TO_CODE.values()) | {CAPITAL_CODE, NUMBER_CODE})
CODE_TO_CHAR = {v:k for k,v in CHAR_TO_CODE.items()}

def hamming_code(a, b):
    return int(np.sum(np.array(a) != np.array(b)))

def char_to_code(ch):
    return CHAR_TO_CODE.get(ch.lower(), None)

def encode_text_to_cells(text):
    cells = []
    for ch in text.lower():
        c = char_to_code(ch)
        if c is not None:
            cells.append(c)
    return cells

def cells_to_text(cells):
    out, cap, number = [], False, False
    digit_map = {"a":"1","b":"2","c":"3","d":"4","e":"5","f":"6","g":"7","h":"8","i":"9","j":"0"}
    for c in cells:
        if c == CAPITAL_CODE:
            cap = True
            continue
        if c == NUMBER_CODE:
            number = True
            continue
        ch = CODE_TO_CHAR.get(tuple(c), "□")
        if number and ch in digit_map:
            ch = digit_map[ch]
            number = False
        if cap and ch.isalpha():
            ch = ch.upper()
            cap = False
        out.append(ch)
    return "".join(out)

print("Canonical cells:", len(ALL_CODES), "| Known characters:", len(CHAR_TO_CODE))


In [ ]:

# 4. Data structures and real/synthetic loaders
# Purpose: create a common sample representation for image and tactile modalities.

@dataclass
class Sample:
    uid: str
    modality: str
    x: np.ndarray
    code: tuple
    group: str
    sequence: tuple

def code_key(code):
    return "".join(map(str, code))

def code_from_name(name):
    name = str(name).strip().lower()
    if len(name) == 1 and name in CHAR_TO_CODE:
        return CHAR_TO_CODE[name]
    if name in [code_key(c) for c in ALL_CODES]:
        return tuple(map(int, name))
    # common folder conventions: "a", "letter_a", "class_a"
    m = re.search(r"(?:letter[_-]?)?([a-z])$", name)
    if m and m.group(1) in CHAR_TO_CODE:
        return CHAR_TO_CODE[m.group(1)]
    return None

def make_synthetic_image(code, size=96, noise=0.025, seed=0):
    rng = np.random.default_rng(seed)
    img = np.ones((size, size), np.float32) * 0.10
    # six lattice positions
    pts = [(28,25),(48,25),(68,25),(28,55),(48,55),(68,55)]
    for active, (cx,cy) in zip(code, pts):
        if active:
            cv2.circle(img, (cx,cy), 7, 0.95, -1)
            cv2.circle(img, (cx-2,cy-2), 3, 1.0, -1)
    # mild illumination / perspective variation
    alpha = rng.uniform(0.85, 1.15)
    img = np.clip(img * alpha + rng.normal(0, noise, img.shape), 0, 1)
    return img.astype(np.float32)

def make_synthetic_tactile(code, length=256, channels=12, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(length)
    x = rng.normal(0, 0.03, (channels, length)).astype(np.float32)
    active = np.where(np.array(code) > 0)[0]
    center = rng.integers(80, 175)
    width = rng.integers(8, 20)
    pulse = np.exp(-0.5*((t-center)/width)**2)
    for d in active:
        ch = int((d*2 + d//3) % channels)
        x[ch] += pulse * rng.uniform(0.7, 1.2)
    return x.astype(np.float32)

def generate_demo_samples(n_per_code=35):
    # A compact subset of the canonical alphabet keeps demo training fast.
    codes = [CHAR_TO_CODE[c] for c in list("abcdefghijklmnopqrstuvwxyz")[:16]]
    images, tactile = [], []
    for ci, code in enumerate(codes):
        for j in range(n_per_code):
            group = f"img_source_{ci}_{j//5}"
            images.append(Sample(f"img_{ci}_{j}", "image",
                                  make_synthetic_image(code, seed=1000+ci*100+j),
                                  code, group, (code,)))
            group_t = f"tactile_session_{ci}_{j//5}"
            tactile.append(Sample(f"tac_{ci}_{j}", "tactile",
                                  make_synthetic_tactile(code, seed=2000+ci*100+j),
                                  code, group_t, (code,)))
    return images, tactile

def load_real_images(root):
    root = Path(root)
    samples = []
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() not in {".jpg",".jpeg",".png",".bmp",".tif",".tiff"}:
            continue
        code = code_from_name(p.parent.name)
        if code is None:
            continue
        im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if im is None: 
            continue
        im = cv2.resize(im, (CFG.image_size, CFG.image_size)).astype(np.float32)/255.0
        # Use parent/source lineage as the group when no metadata is available.
        group = p.stem.split("_aug")[0].split("-aug")[0]
        samples.append(Sample(p.stem, "image", im, code, group, (code,)))
    return samples

def load_real_tactile(root):
    root = Path(root)
    samples = []
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() not in {".pkl",".pickle",".npy",".npz"}:
            continue
        try:
            if p.suffix.lower() in {".pkl",".pickle"}:
                obj = pickle.load(open(p,"rb"))
            elif p.suffix.lower()==".npy":
                obj = np.load(p, allow_pickle=True)
            else:
                z = np.load(p, allow_pickle=True)
                obj = z[list(z.keys())[0]]
        except Exception:
            continue
        code = code_from_name(p.parent.name)
        if code is None:
            continue
        arr = np.asarray(obj)
        if arr.ndim == 1:
            arr = arr[None,:]
        if arr.ndim != 2:
            continue
        if arr.shape[0] != CFG.tactile_channels and arr.shape[1] == CFG.tactile_channels:
            arr = arr.T
        if arr.shape[0] != CFG.tactile_channels:
            continue
        if arr.shape[1] != CFG.tactile_len:
            old = np.linspace(0,1,arr.shape[1])
            new = np.linspace(0,1,CFG.tactile_len)
            arr = np.vstack([np.interp(new, old, row) for row in arr])
        samples.append(Sample(p.stem, "tactile", arr.astype(np.float32), code,
                              p.stem, (code,)))
    return samples

if CFG.demo_mode:
    image_samples, tactile_samples = generate_demo_samples()
else:
    image_samples = load_real_images(CFG.image_root)
    tactile_samples = load_real_tactile(CFG.tactile_root)

print("Images:", len(image_samples), "Tactile:", len(tactile_samples))
if not image_samples or not tactile_samples:
    raise RuntimeError("No usable data found. Set DEMO_MODE=True or configure real dataset paths/label folders.")


In [ ]:

# 5. Group-aware train/validation/test partitioning
# Purpose: prevent source-image/session leakage across partitions.

def group_split(samples, seed=42):
    idx = np.arange(len(samples))
    groups = np.array([s.group for s in samples])
    y = np.array([code_key(s.code) for s in samples])
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    tr_idx, tmp_idx = next(gss1.split(idx, y, groups))
    tmp_groups = groups[tmp_idx]
    tmp_y = y[tmp_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed+1)
    va_rel, te_rel = next(gss2.split(tmp_idx, tmp_y, tmp_groups))
    return idx[tr_idx], tmp_idx[va_rel], tmp_idx[te_rel]

img_tr, img_va, img_te = group_split(image_samples, CFG.seed)
tac_tr, tac_va, tac_te = group_split(tactile_samples, CFG.seed+11)

print("Image split:", len(img_tr), len(img_va), len(img_te))
print("Tactile split:", len(tac_tr), len(tac_va), len(tac_te))

def overlap_groups(a,b):
    return len(set(s.group for s in a) & set(s.group for s in b))

assert overlap_groups([image_samples[i] for i in img_tr], [image_samples[i] for i in img_te]) == 0
assert overlap_groups([tactile_samples[i] for i in tac_tr], [tactile_samples[i] for i in tac_te]) == 0
print("Leakage check: PASS")


In [ ]:

# 6. URC-APG: preprocessing routes + Braille-specific quality descriptors
# Purpose: implement minimal/standard/enhanced routes and a low-cost task-aware quality vector.

def image_route(x, route):
    x = np.asarray(x, np.float32)
    if route == 0:
        y = cv2.resize(x, (CFG.image_size, CFG.image_size))
        y = cv2.GaussianBlur(y, (3,3), 0)
        _, y = cv2.threshold((y*255).astype(np.uint8), 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        return y.astype(np.float32)/255.0
    if route == 1:
        y = (x*255).astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        y = clahe.apply(y)
        y = cv2.GaussianBlur(y, (3,3), 0)
        _, y = cv2.threshold(y, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        return y.astype(np.float32)/255.0
    y = image_route(x, 1)
    kernel = np.ones((3,3), np.uint8)
    y = cv2.morphologyEx((y*255).astype(np.uint8), cv2.MORPH_OPEN, kernel)
    return y.astype(np.float32)/255.0

def tactile_route(x, route):
    x = np.asarray(x, np.float32)
    if route == 0:
        lo, hi = np.percentile(x, 2), np.percentile(x, 98)
        return np.clip((x-lo)/(hi-lo+1e-8),0,1)
    # Savitzky-Golay replacement without scipy.signal dependency: moving average.
    kernel = np.ones(7, np.float32)/7
    y = np.vstack([np.convolve(row, kernel, mode="same") for row in x])
    if route == 2:
        med = np.median(y, axis=1, keepdims=True)
        mad = np.median(np.abs(y-med), axis=1, keepdims=True)+1e-6
        y = np.clip(y, med-5*mad, med+5*mad)
    lo, hi = np.percentile(y, 2), np.percentile(y, 98)
    return np.clip((y-lo)/(hi-lo+1e-8),0,1)

def image_quality(x):
    x = np.asarray(x, np.float32)
    contrast = float(np.std(x))
    sharp = float(cv2.Laplacian(x, cv2.CV_32F).var())
    # Lattice-like autocorrelation peaks at short horizontal/vertical offsets.
    z = x - x.mean()
    ac_h = np.mean(z[:, :-8] * z[:, 8:]) / (z.var()+1e-8)
    ac_v = np.mean(z[:-8, :] * z[8:, :]) / (z.var()+1e-8)
    # Dot separability proxy: bright/dark separation around detected blobs.
    th = np.percentile(x, 90)
    mask = x >= th
    sep = float((x[mask].mean() - x[~mask].mean()) / (x[mask].std()+x[~mask].std()+1e-6)) if mask.any() else 0.0
    return np.array([contrast, sharp, (ac_h+ac_v)/2, sep], np.float32)

def tactile_quality(x):
    x = np.asarray(x, np.float32)
    corr = np.corrcoef(x)
    off = corr[np.triu_indices_from(corr, 1)]
    coherence = float(np.nanmean(np.nan_to_num(off))) if len(off) else 0
    baseline = x[:max(10, x.shape[1]//8)]
    cv = float(np.mean(np.std(baseline,axis=1)/(np.abs(np.mean(baseline,axis=1))+1e-5)))
    smooth = np.vstack([np.convolve(r, np.ones(7)/7, mode="same") for r in x])
    residual = x-smooth
    snr = float(10*np.log10((np.var(smooth)+1e-8)/(np.var(residual)+1e-8)))
    jitter = float(np.std(np.argmax(np.abs(smooth),axis=1)))
    return np.array([coherence, jitter, cv, snr], np.float32)

def fit_percentile_scaler(qs):
    qs = np.asarray(qs)
    lo = np.percentile(qs, 5, axis=0)
    hi = np.percentile(qs, 95, axis=0)
    return lo, hi

def scale_quality(q, lo, hi):
    return np.clip((q-lo)/(hi-lo+1e-8), 0, 1).astype(np.float32)

def build_quality(samples):
    return np.vstack([image_quality(s.x) if s.modality=="image" else tactile_quality(s.x) for s in samples])

img_q_lo, img_q_hi = fit_percentile_scaler(build_quality([image_samples[i] for i in img_tr]))
tac_q_lo, tac_q_hi = fit_percentile_scaler(build_quality([tactile_samples[i] for i in tac_tr]))

print("URC-APG route costs:", CFG.route_costs)


In [ ]:

# 7. Utility predictor + empirical risk-controlled routing
# Purpose: learn route utility from quality descriptors and select a route under a cost constraint.

class UtilityPredictor(nn.Module):
    def __init__(self, in_dim=4, routes=3, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(.1),
            nn.Linear(hidden, hidden), nn.ReLU()
        )
        self.gain = nn.Linear(hidden, routes)
        self.rel = nn.Linear(hidden, routes)
    def forward(self, q):
        h = self.net(q)
        return self.gain(h), torch.sigmoid(self.rel(h))

def quick_reference_correctness(samples, modality, qlo, qhi):
    # Reference correctness proxy used to generate counterfactual training targets.
    # For real experiments, replace this with the manuscript's cross-fitted reference recognizers.
    out = []
    for s in samples:
        q = scale_quality(image_quality(s.x) if modality=="image" else tactile_quality(s.x), qlo, qhi)
        base = float(q.mean())
        route_acc = []
        for r in range(3):
            # Higher quality + route-specific benefit; noisy but reproducible target.
            benefit = [0.00, 0.06, 0.11][r]
            score = np.clip(base + benefit*(1-base), 0, 1)
            route_acc.append(score)
        out.append(route_acc)
    return np.asarray(out)

def fit_utility_predictor(samples, modality, qlo, qhi, epochs=80):
    q = build_quality(samples)
    q = np.vstack([scale_quality(v,qlo,qhi) for v in q])
    targets = quick_reference_correctness(samples, modality, qlo, qhi)
    base = targets[:,0:1]
    gains = targets-base
    rel = targets > 0.65
    model = UtilityPredictor(4,3,CFG.utility_hidden).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    qt = torch.tensor(q, dtype=torch.float32, device=DEVICE)
    gt = torch.tensor(gains, dtype=torch.float32, device=DEVICE)
    rt = torch.tensor(rel, dtype=torch.float32, device=DEVICE)
    for _ in range(epochs):
        pred_g, pred_r = model(qt)
        loss = F.mse_loss(pred_g, gt) + F.binary_cross_entropy(pred_r, rt)
        opt.zero_grad(); loss.backward(); opt.step()
    return model

img_util = fit_utility_predictor([image_samples[i] for i in img_tr], "image", img_q_lo, img_q_hi)
tac_util = fit_utility_predictor([tactile_samples[i] for i in tac_tr], "tactile", tac_q_lo, tac_q_hi)

def select_routes(samples, modality, qlo, qhi, model, lam=0.05):
    model.eval()
    qraw = build_quality(samples)
    q = np.vstack([scale_quality(v,qlo,qhi) for v in qraw])
    with torch.no_grad():
        gains, rel = model(torch.tensor(q,dtype=torch.float32,device=DEVICE))
        gains, rel = gains.cpu().numpy(), rel.cpu().numpy()
    score = gains*rel - lam*np.array(CFG.route_costs)[None,:]
    route = np.argmax(score, axis=1)
    quality = rel[np.arange(len(route)), route]
    return route, quality, q, gains, rel

img_routes, img_rel, _, _, _ = select_routes(
    [image_samples[i] for i in img_tr], "image", img_q_lo, img_q_hi, img_util, 0.05
)
tac_routes, tac_rel, _, _, _ = select_routes(
    [tactile_samples[i] for i in tac_tr], "tactile", tac_q_lo, tac_q_hi, tac_util, 0.05
)

print("Training route distribution - image:", Counter(img_routes))
print("Training route distribution - tactile:", Counter(tac_routes))


In [ ]:

# 8. Dataset tensors with controlled degradation
# Purpose: expose visual, tactile, quality and availability information to the multimodal model.

def degrade_image(x, severity, seed=0):
    if severity == 0: return x.copy()
    rng = np.random.default_rng(seed)
    y = x.copy()
    if severity >= 1:
        y = cv2.GaussianBlur(y, (5,5), 1.0)
        y = np.clip(y ** rng.uniform(1.2,1.8),0,1)
        y = np.clip(y + rng.normal(0,0.04,y.shape),0,1)
    if severity >= 2:
        M = cv2.getRotationMatrix2D((y.shape[1]/2,y.shape[0]/2), rng.uniform(-5,5), 1)
        y = cv2.warpAffine(y, M, (y.shape[1],y.shape[0]), borderValue=0.1)
        y = np.clip(y + rng.normal(0,0.08,y.shape),0,1)
    return y.astype(np.float32)

def degrade_tactile(x, severity, seed=0):
    if severity == 0: return x.copy()
    rng = np.random.default_rng(seed)
    y = x.copy()
    if severity >= 1:
        y += rng.normal(0,0.06,y.shape).astype(np.float32)
        drop = rng.random(y.shape[0]) < 0.12
        y[drop] = 0
    if severity >= 2:
        drift = np.linspace(0, rng.uniform(-0.3,0.3), y.shape[1], dtype=np.float32)
        y += drift[None,:]
        drop = rng.random(y.shape[0]) < 0.25
        y[drop] = 0
    return y.astype(np.float32)

class BraillePairDataset(Dataset):
    def __init__(self, images, tactile, split="train", seed=42):
        self.images, self.tactile = images, tactile
        self.split, self.seed = split, seed
        self.codes = sorted(set([s.code for s in images+tactile]), key=code_key)
        self.code_to_idx = {c:i for i,c in enumerate(self.codes)}
        self.tac_by_code = defaultdict(list)
        for i,s in enumerate(tactile): self.tac_by_code[s.code].append(i)

    def __len__(self):
        return max(len(self.images), len(self.tactile))

    def __getitem__(self, idx):
        rng = np.random.default_rng(self.seed + idx)
        im = self.images[idx % len(self.images)]
        code = im.code
        ti = rng.choice(self.tac_by_code[code])
        ta = self.tactile[ti]
        # Training can sample availability/degradation; validation/test are fixed.
        if self.split == "train":
            av_i = bool(rng.random() > 0.15)
            av_t = bool(rng.random() > 0.15)
            if not av_i and not av_t: av_i = True
            si = int(rng.integers(0,3))
            st = int(rng.integers(0,3))
        else:
            av_i, av_t, si, st = True, True, 0, 0
        xi = degrade_image(im.x, si, seed=self.seed+idx) if av_i else np.zeros_like(im.x)
        xt = degrade_tactile(ta.x, st, seed=self.seed+idx) if av_t else np.zeros_like(ta.x)
        qi = image_quality(xi); qt = tactile_quality(xt)
        qi = scale_quality(qi,img_q_lo,img_q_hi)
        qt = scale_quality(qt,tac_q_lo,tac_q_hi)
        # Route using utility models.
        ri, rti = 1, 1
        if av_i:
            ri = int(select_routes([Sample("x","image",xi,code,"x",(code,))],"image",img_q_lo,img_q_hi,img_util,0.05)[0][0])
            xi = image_route(xi,ri)
        if av_t:
            rti = int(select_routes([Sample("x","tactile",xt,code,"x",(code,))],"tactile",tac_q_lo,tac_q_hi,tac_util,0.05)[0][0])
            xt = tactile_route(xt,rti)
        return {
            "image": torch.tensor(xi[None],dtype=torch.float32),
            "tactile": torch.tensor(xt,dtype=torch.float32),
            "qi": torch.tensor(float(np.mean(qi)),dtype=torch.float32),
            "qt": torch.tensor(float(np.mean(qt)),dtype=torch.float32),
            "avail": torch.tensor([float(av_i),float(av_t)],dtype=torch.float32),
            "label": torch.tensor(self.code_to_idx[code],dtype=torch.long),
            "code": code
        }

img_train = [image_samples[i] for i in img_tr]
img_val   = [image_samples[i] for i in img_va]
img_test  = [image_samples[i] for i in img_te]
tac_train = [tactile_samples[i] for i in tac_tr]
tac_val   = [tactile_samples[i] for i in tac_va]
tac_test  = [tactile_samples[i] for i in tac_te]

train_ds = BraillePairDataset(img_train, tac_train, "train", 42)
val_ds   = BraillePairDataset(img_val, tac_val, "val", 43)
test_ds  = BraillePairDataset(img_test, tac_test, "test", 44)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=0)

NUM_CLASSES = len(train_ds.codes)
print("Shared alphabet/classes:", NUM_CLASSES)


In [ ]:

# 9. BrailleMamba-Lite and TactileShuffle-Lite
# Purpose: implement lightweight modality-specific structured encoders.

class BrailleMambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.decay = nn.Linear(dim, dim)
        self.inp = nn.Linear(dim, dim)
        self.out = nn.Linear(dim, dim)
        self.ff = nn.Sequential(nn.Linear(dim, dim*2), nn.GELU(), nn.Linear(dim*2, dim))
    def scan(self, x):
        h = torch.zeros(x.size(0), x.size(2), device=x.device)
        ys=[]
        for t in range(x.size(1)):
            z=x[:,t]
            a=torch.sigmoid(self.decay(z))
            b=self.inp(z)
            h=a*h+(1-a)*b
            ys.append(self.out(h))
        return torch.stack(ys,1)
    def forward(self,x):
        y=self.scan(self.norm(x))
        yr=torch.flip(self.scan(torch.flip(self.norm(x),[1])),[1])
        return x + 0.5*(y+yr) + self.ff(self.norm(x))

class BrailleMambaLite(nn.Module):
    def __init__(self, embed=128, token_dim=64, blocks=2):
        super().__init__()
        self.stem=nn.Sequential(
            nn.Conv2d(1,32,3,2,1), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32,token_dim,3,2,1), nn.BatchNorm2d(token_dim), nn.GELU(),
            nn.Conv2d(token_dim,token_dim,3,2,1), nn.BatchNorm2d(token_dim), nn.GELU()
        )
        self.blocks=nn.ModuleList([BrailleMambaBlock(token_dim) for _ in range(blocks)])
        self.proj=nn.Linear(token_dim,embed)
        self.seq_proj=nn.Linear(token_dim*12,embed)
    def forward(self,x):
        fmap=self.stem(x)
        b,c,h,w=fmap.shape
        tok=fmap.flatten(2).transpose(1,2)
        for blk in self.blocks: tok=blk(tok)
        emb=self.proj(tok).mean(1)
        # Order-preserving vertical fusion for width-wise CTC sequence.
        v=tok.transpose(1,2).reshape(b,c,h,w).permute(0,3,1,2).reshape(b,w,c*h)
        seq=self.seq_proj(v)
        return emb,tok,seq,fmap

class TactileShuffleBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        hidden=c*2
        self.pw1=nn.Conv1d(c,hidden,1)
        self.dw=nn.Conv1d(hidden,hidden,5,padding=2,groups=4)
        self.pw2=nn.Conv1d(hidden,c,1)
        self.bn=nn.BatchNorm1d(c)
    def forward(self,x):
        y=F.gelu(self.pw1(x))
        y=F.gelu(self.dw(y))
        y=self.pw2(y)
        return F.gelu(self.bn(y+x))

class TactileShuffleLite(nn.Module):
    def __init__(self, channels=12, embed=128, blocks=2):
        super().__init__()
        self.stem=nn.Sequential(
            nn.Conv1d(channels,48,5,2,2),nn.BatchNorm1d(48),nn.GELU(),
            nn.Conv1d(48,64,5,2,2),nn.BatchNorm1d(64),nn.GELU()
        )
        self.blocks=nn.ModuleList([TactileShuffleBlock(64) for _ in range(blocks)])
        self.proj=nn.Linear(64,embed)
        self.att=nn.Linear(64,1)
    def forward(self,x):
        y=self.stem(x)
        for blk in self.blocks:y=blk(y)
        tok=y.transpose(1,2)
        z=self.proj(tok)
        a=torch.softmax(self.att(tok).squeeze(-1),dim=1)
        emb=(z*a.unsqueeze(-1)).sum(1)
        return emb,tok,a

print("Encoders ready.")


In [ ]:

# 10. BCAN: cross-modal context, shared/private factorization, prototypes and reliability fusion
# Purpose: align and fuse modalities while tolerating missing/degraded inputs.

class BCAN(nn.Module):
    def __init__(self, dim=128, classes=16, heads=4):
        super().__init__()
        self.att_i=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.att_t=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.norm=nn.LayerNorm(dim)
        self.shared=nn.Linear(dim,dim//2)
        self.private=nn.Linear(dim,dim//2)
        self.gate=nn.Sequential(
            nn.Linear(dim//2+3,64),nn.ReLU(),nn.Linear(64,1)
        )
        self.head=nn.Linear(dim//2,classes)
    def forward(self, ei, ti, qi, qt, avail, prototypes=None):
        # Inputs are pooled embeddings; singleton token context is used here.
        it=ei.unsqueeze(1); tt=ti.unsqueeze(1)
        if avail[:,0].sum()>0 and avail[:,1].sum()>0:
            ii,_=self.att_i(it,tt,tt)
            tt2,_=self.att_t(tt,it,it)
            ei2=self.norm(ei+ii[:,0])
            ti2=self.norm(ti+tt2[:,0])
        else:
            ei2,ti2=ei,ti
        si,pi=self.shared(ei2),self.private(ei2)
        st,pt=self.shared(ti2),self.private(ti2)

        proto_ei=torch.zeros(ei.size(0),device=ei.device)
        proto_et=torch.zeros(ei.size(0),device=ei.device)
        if prototypes is not None:
            for k,p in prototypes.items():
                mask = torch.tensor(k,device=ei.device).reshape(1)
        fi=torch.cat([si,qi.unsqueeze(1),torch.zeros_like(qi.unsqueeze(1)),avail[:,0:1]],1)
        ft=torch.cat([st,qt.unsqueeze(1),torch.zeros_like(qt.unsqueeze(1)),avail[:,1:2]],1)
        gi=self.gate(fi).squeeze(1)
        gt=self.gate(ft).squeeze(1)
        gi=gi.masked_fill(avail[:,0]<0.5,-1e9)
        gt=gt.masked_fill(avail[:,1]<0.5,-1e9)
        w=torch.softmax(torch.stack([gi,gt],1),1)
        u=w[:,0:1]*si+w[:,1:2]*st
        logits=self.head(u)
        return {"u":u,"si":si,"st":st,"pi":pi,"pt":pt,"weights":w,
                "logits":logits,"ei":ei2,"et":ti2}

class ReviewEstimator(nn.Module):
    def __init__(self, in_dim=8, hidden=64):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden),nn.ReLU(),nn.Dropout(.1),
                               nn.Linear(hidden,1),nn.Sigmoid())
    def forward(self,x): return self.net(x).squeeze(-1)

class FullBrailleModel(nn.Module):
    def __init__(self, classes):
        super().__init__()
        self.visual=BrailleMambaLite(CFG.embed_dim,CFG.token_dim,CFG.visual_blocks)
        self.tactile=TactileShuffleLite(CFG.tactile_channels,CFG.embed_dim,CFG.tactile_blocks)
        self.bcan=BCAN(CFG.embed_dim,classes)
        self.ctc=nn.Linear(CFG.embed_dim,classes+1)
    def forward(self,image,tactile,qi,qt,avail):
        ei,itok,seq,fmap=self.visual(image)
        et,ttok,tatt=self.tactile(tactile)
        b=self.bcan(ei,et,qi,qt,avail)
        b["ctc_logits"]=self.ctc(seq)
        b["visual_tokens"]=itok; b["tactile_tokens"]=ttok
        b["fmap"]=fmap; b["tactile_attention"]=tatt
        return b

model=FullBrailleModel(NUM_CLASSES).to(DEVICE)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:

# 11. Prototype computation + training
# Purpose: train recognition, shared/private separation, prototype alignment and missing-modality consistency.

def compute_prototypes(model, loader):
    model.eval()
    acc=defaultdict(list)
    with torch.no_grad():
        for batch in loader:
            image=batch["image"].to(DEVICE); tactile=batch["tactile"].to(DEVICE)
            qi=batch["qi"].to(DEVICE); qt=batch["qt"].to(DEVICE); av=batch["avail"].to(DEVICE)
            out=model(image,tactile,qi,qt,av)
            for j,c in enumerate(batch["code"]):
                acc[c].append(F.normalize(out["u"][j],dim=0).cpu())
    return {c:F.normalize(torch.stack(v).mean(0),dim=0).to(DEVICE) for c,v in acc.items()}

def prototype_loss(out, codes, prototypes):
    vals=[]
    for i,c in enumerate(codes):
        if c in prototypes:
            vals.append(1-F.cosine_similarity(out["u"][i].unsqueeze(0),prototypes[c].unsqueeze(0)).mean())
    return torch.stack(vals).mean() if vals else out["u"].sum()*0

def train_model(model, train_loader, val_loader):
    opt=torch.optim.AdamW(model.parameters(),lr=CFG.lr,weight_decay=CFG.weight_decay)
    best_state=None; best=-1; bad=0; history=[]
    for epoch in range(CFG.epochs):
        model.train()
        running=0
        for batch in train_loader:
            image=batch["image"].to(DEVICE); tactile=batch["tactile"].to(DEVICE)
            qi=batch["qi"].to(DEVICE); qt=batch["qt"].to(DEVICE); av=batch["avail"].to(DEVICE)
            y=batch["label"].to(DEVICE)
            out=model(image,tactile,qi,qt,av)
            loss_cls=F.cross_entropy(out["logits"],y)
            # Shared/private decorrelation and private non-collapse.
            cov=torch.cov(torch.cat([out["si"],out["pi"]],1).T)
            d=out["si"].shape[1]
            cross=cov[:d,d:]
            loss_orth=(cross**2).mean()
            loss_var=((out["pi"].std(0)-1)**2).mean()
            loss=loss_cls+0.02*loss_orth+0.01*loss_var
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),CFG.grad_clip)
            opt.step()
            running += float(loss.item())
        # Validation
        model.eval(); ys=[]; ps=[]
        with torch.no_grad():
            for batch in val_loader:
                out=model(batch["image"].to(DEVICE),batch["tactile"].to(DEVICE),
                          batch["qi"].to(DEVICE),batch["qt"].to(DEVICE),batch["avail"].to(DEVICE))
                ys.extend(batch["label"].numpy().tolist())
                ps.extend(out["logits"].argmax(1).cpu().numpy().tolist())
        acc=accuracy_score(ys,ps)
        history.append((epoch+1,running/max(1,len(train_loader)),acc))
        print(f"Epoch {epoch+1:02d} | loss={history[-1][1]:.4f} | val_acc={acc:.4f}")
        if acc>best:
            best=acc; best_state=copy.deepcopy(model.state_dict()); bad=0
        else:
            bad+=1
            if bad>=CFG.patience: break
    model.load_state_dict(best_state)
    return pd.DataFrame(history,columns=["epoch","loss","val_accuracy"])

history=train_model(model,train_loader,val_loader)
display(history.tail())


In [ ]:

# 12. Evaluation under availability and degradation
# Purpose: measure fused, image-only and tactile-only behavior without inventing target numbers.

def evaluate(model, loader, availability="fused", severity=0):
    model.eval(); y=[]; p=[]; probs=[]; weights=[]
    with torch.no_grad():
        for batch in loader:
            image=batch["image"].clone(); tactile=batch["tactile"].clone()
            if severity:
                image=torch.stack([torch.tensor(degrade_image(x[0].numpy(),severity,123+i)) for i,x in enumerate(image)]).unsqueeze(1)
                tactile=torch.stack([torch.tensor(degrade_tactile(x.numpy(),severity,321+i)) for i,x in enumerate(tactile)])
            av=batch["avail"].clone()
            if availability=="image": av[:,1]=0
            if availability=="tactile": av[:,0]=0
            out=model(image.to(DEVICE),tactile.to(DEVICE),batch["qi"].to(DEVICE),
                      batch["qt"].to(DEVICE),av.to(DEVICE))
            pr=F.softmax(out["logits"],1)
            y.extend(batch["label"].numpy()); p.extend(pr.argmax(1).cpu().numpy())
            probs.extend(pr.cpu().numpy()); weights.extend(out["weights"].cpu().numpy())
    return {
        "accuracy":accuracy_score(y,p),
        "macro_f1":f1_score(y,p,average="macro"),
        "y":np.array(y), "p":np.array(p), "probs":np.array(probs),
        "weights":np.array(weights)
    }

rows=[]
for severity in [0,1,2]:
    for mode in ["fused","image","tactile"]:
        r=evaluate(model,test_loader,mode,severity)
        rows.append({"severity":severity,"availability":mode,
                     "accuracy":r["accuracy"],"macro_f1":r["macro_f1"]})
eval_df=pd.DataFrame(rows)
display(eval_df)


In [ ]:

# 13. BRNEN-CG: confidence estimator and risk-controlled selective review
# Purpose: estimate cell correctness and select a threshold using validation data only.

def collect_review_features(model, loader):
    model.eval(); feats=[]; correct=[]; post=[]
    with torch.no_grad():
        for batch in loader:
            out=model(batch["image"].to(DEVICE),batch["tactile"].to(DEVICE),
                      batch["qi"].to(DEVICE),batch["qt"].to(DEVICE),batch["avail"].to(DEVICE))
            prob=F.softmax(out["logits"],1)
            mx,pr=prob.max(1)
            w=out["weights"]
            agree=(out["logits"].argmax(1)==out["logits"].argmax(1)).float()
            f=torch.stack([
                mx,batch["qi"].to(DEVICE),batch["qt"].to(DEVICE),
                w[:,0],w[:,1],agree,
                torch.max(w,1).values, torch.min(w,1).values
            ],1)
            feats.append(f.cpu().numpy())
            correct.append((pr.cpu().numpy()==batch["label"].numpy()).astype(float))
            post.append(mx.cpu().numpy())
    return np.vstack(feats), np.concatenate(correct), np.concatenate(post)

review_X, review_y, review_post = collect_review_features(model,val_loader)
review_est=ReviewEstimator(8,CFG.review_hidden).to(DEVICE)
opt=torch.optim.AdamW(review_est.parameters(),lr=2e-3)
X=torch.tensor(review_X,dtype=torch.float32,device=DEVICE)
Y=torch.tensor(review_y,dtype=torch.float32,device=DEVICE)
for _ in range(120):
    pred=review_est(X)
    loss=F.binary_cross_entropy(pred,Y)
    opt.zero_grad();loss.backward();opt.step()

with torch.no_grad():
    conf=review_est(X).cpu().numpy()

def choose_review_threshold(conf,y,thresholds,alpha=0.02):
    candidates=[]
    for t in thresholds:
        accepted=conf>=t
        if accepted.sum()==0:
            err=0
        else:
            err=1-y[accepted].mean()
        review_rate=1-accepted.mean()
        candidates.append((t,err,review_rate))
    feasible=[z for z in candidates if z[1] <= alpha]
    return min(feasible,key=lambda z:(z[2],-z[0])) if feasible else (1.01,0,1.0)

review_threshold, accepted_error, review_rate = choose_review_threshold(
    conf,review_y,CFG.review_thresholds,CFG.review_error_tolerance
)
print("Selected validation threshold:", review_threshold)
print("Validation accepted-cell error:", accepted_error)
print("Validation review rate:", review_rate)

# Test policy (fixed before test evaluation)
test_X,test_y,test_post=collect_review_features(model,test_loader)
with torch.no_grad():
    test_conf=review_est(torch.tensor(test_X,dtype=torch.float32,device=DEVICE)).cpu().numpy()
accepted=test_conf>=review_threshold
test_error=1-test_y[accepted].mean() if accepted.any() else 0
print("Test accepted-cell error:", round(float(test_error),4))
print("Test review rate:", round(float(1-accepted.mean()),4))


In [ ]:

# 14. Accessible deterministic note-taking layer
# Purpose: convert confirmed Grade-1 Braille cells into editable structured text and accessibility outputs.

def textrank_keypoints(text, top_k=3):
    sentences=[s.strip() for s in re.split(r'(?<=[.!?])\s+',text) if s.strip()]
    if len(sentences)<=top_k: return sentences
    vec=TfidfVectorizer(stop_words="english")
    X=vec.fit_transform(sentences).toarray()
    sim=cosine_similarity(X)
    n=len(sentences); score=np.ones(n)/n
    for _ in range(50):
        score=0.15/n+0.85*(sim/sim.sum(1,keepdims=True).clip(min=1e-8)).T@score
    idx=np.argsort(score)[::-1][:top_k]
    return [sentences[i] for i in sorted(idx)]

def key_terms(text, top_k=5):
    vec=TfidfVectorizer(stop_words="english")
    X=vec.fit_transform([text])
    if X.shape[1]==0:return []
    terms=np.array(vec.get_feature_names_out())
    vals=X.toarray()[0]
    return terms[np.argsort(vals)[::-1][:top_k]].tolist()

def cells_to_unicode_braille(cells):
    chars=[]
    for c in cells:
        bits=sum((1<<i) for i,v in enumerate(c) if v)
        chars.append(chr(0x2800+bits))
    return "".join(chars)

def build_structured_note(cells):
    text=cells_to_text(cells)
    return {
        "content":text,
        "key_points":textrank_keypoints(text,3) if text else [],
        "key_terms":key_terms(text,5) if text else [],
        "dominant_term":key_terms(text,1)[0] if text and key_terms(text,1) else "",
        "unicode_braille":cells_to_unicode_braille(cells),
        "source_cell_count":len(cells)
    }

example_text="education improves access to information. accessible notes support independent learning."
example_cells=encode_text_to_cells(example_text)
note=build_structured_note(example_cells)
print(json.dumps(note,indent=2,ensure_ascii=False))


In [ ]:

# 15. Explainability: Grad-CAM for visual stem + integrated gradients for tactile input
# Purpose: produce spatial visual evidence and taxel-time attribution maps.

def gradcam_visual(model, image, tactile, qi, qt, avail, class_idx=None):
    model.eval()
    image=image.clone().detach().to(DEVICE).requires_grad_(True)
    tactile=tactile.to(DEVICE)
    qi,qt,avail=qi.to(DEVICE),qt.to(DEVICE),avail.to(DEVICE)
    saved={}
    def hook(m,inp,out): saved["fmap"]=out
    h=model.visual.stem.register_forward_hook(hook)
    out=model(image,tactile,qi,qt,avail)
    if class_idx is None: class_idx=int(out["logits"].argmax(1)[0])
    score=out["logits"][0,class_idx]
    model.zero_grad(); score.backward()
    fmap=saved["fmap"]
    grad=fmap.grad if fmap.requires_grad else None
    if grad is None:
        # retain gradient by a second forward hook
        pass
    h.remove()
    # Robust fallback: gradient of the stem output through explicit retained graph.
    # Recompute with retain_grad.
    model.zero_grad()
    saved={}
    def hook2(m,inp,out):
        out.retain_grad(); saved["fmap"]=out
    h=model.visual.stem.register_forward_hook(hook2)
    out=model(image,tactile,qi,qt,avail)
    score=out["logits"][0,class_idx]
    score.backward()
    fmap=saved["fmap"]; grad=fmap.grad
    w=grad.mean(dim=(2,3),keepdim=True)
    cam=F.relu((w*fmap).sum(1,keepdim=True))
    cam=F.interpolate(cam,size=image.shape[-2:],mode="bilinear",align_corners=False)
    cam=cam[0,0].detach().cpu().numpy()
    cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8)
    h.remove()
    return cam

def integrated_gradients_tactile(model, image, tactile, qi, qt, avail, steps=24):
    model.eval()
    base=torch.zeros_like(tactile)
    alphas=torch.linspace(0,1,steps,device=DEVICE)
    total=torch.zeros_like(tactile,device=DEVICE)
    target=None
    for a in alphas:
        x=(base+a*(tactile-base)).detach().requires_grad_(True)
        out=model(image,x,qi,qt,avail)
        if target is None: target=int(out["logits"].argmax(1)[0])
        score=out["logits"][0,target]
        model.zero_grad(); score.backward()
        total += x.grad.detach()
    avg=total/steps
    attr=(tactile-base)*avg
    return attr[0].detach().cpu().numpy()

sample=next(iter(test_loader))
img=sample["image"][0:1].to(DEVICE)
tac=sample["tactile"][0:1].to(DEVICE)
qi=sample["qi"][0:1].to(DEVICE); qt=sample["qt"][0:1].to(DEVICE); av=sample["avail"][0:1].to(DEVICE)
cam=gradcam_visual(model,img,tac,qi,qt,av)
ig=integrated_gradients_tactile(model,img,tac,qi,qt,av)

fig,ax=plt.subplots(1,3,figsize=(13,4))
ax[0].imshow(img[0,0].cpu(),cmap="gray"); ax[0].set_title("Braille input"); ax[0].axis("off")
ax[1].imshow(img[0,0].cpu(),cmap="gray"); ax[1].imshow(cam,cmap="jet",alpha=.45); ax[1].set_title("Grad-CAM"); ax[1].axis("off")
ax[2].imshow(np.abs(ig),aspect="auto",origin="lower",cmap="magma"); ax[2].set_title("Tactile Integrated Gradients"); ax[2].set_xlabel("Time"); ax[2].set_ylabel("Taxel")
plt.tight_layout()
plt.show()


In [ ]:

# 16. Deletion faithfulness and Hamming-distance error analysis
# Purpose: quantify whether high-attribution inputs affect the prediction more than random deletion.

def deletion_curve_image(model, image, tactile, qi, qt, avail, fractions=np.linspace(0,.8,9)):
    base=image.clone()
    out0=model(image,tactile,qi,qt,avail)
    target=int(out0["logits"].argmax(1)[0])
    scores=[]
    attr=gradcam_visual(model,image,tactile,qi,qt,avail)
    order=np.argsort(attr.ravel())[::-1]
    flat=base[0,0].flatten()
    for f in fractions:
        x=base.clone()
        n=int(f*len(order))
        x[0,0].flatten()[order[:n]]=0.1
        with torch.no_grad():
            s=F.softmax(model(x,tactile,qi,qt,avail)["logits"],1)[0,target].item()
        scores.append(s)
    return np.array(scores)

fractions=np.linspace(0,.8,9)
try:
    deletion=deletion_curve_image(model,img,tac,qi,qt,av,fractions)
    random_scores=[]
    target=int(model(img,tac,qi,qt,av)["logits"].argmax(1)[0])
    rng=np.random.default_rng(123)
    order=rng.permutation(CFG.image_size*CFG.image_size)
    for f in fractions:
        x=img.clone(); n=int(f*len(order))
        x[0,0].flatten()[order[:n]]=0.1
        with torch.no_grad():
            random_scores.append(F.softmax(model(x,tac,qi,qt,av)["logits"],1)[0,target].item())
    plt.figure(figsize=(6,4))
    plt.plot(fractions,deletion,"o-",label="Attribution-guided deletion")
    plt.plot(fractions,random_scores,"s--",label="Random deletion")
    plt.xlabel("Fraction of image elements removed")
    plt.ylabel("Target-cell posterior")
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Deletion test skipped:",e)

def hamming_error_distribution(true_codes, pred_codes):
    dist=[]
    for a,b in zip(true_codes,pred_codes):
        dist.append(hamming_code(a,b))
    return pd.Series(dist).value_counts(normalize=True).sort_index()

# Map predictions back to canonical codes.
code_list=train_ds.codes
test_res=evaluate(model,test_loader,"fused",0)
true_codes=[code_list[i] for i in test_res["y"]]
pred_codes=[code_list[i] for i in test_res["p"]]
hd=hamming_error_distribution(true_codes,pred_codes)
display(hd.rename("error_proportion").to_frame())


In [ ]:

# 17. Ablation and computational profile helpers
# Purpose: provide reproducible hooks for the manuscript's ablation/efficiency tables.

def parameter_count(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def benchmark_model(model, loader, n_batches=10):
    model.eval()
    times=[]
    with torch.no_grad():
        for i,batch in enumerate(loader):
            if i>=n_batches: break
            image=batch["image"].to(DEVICE); tactile=batch["tactile"].to(DEVICE)
            qi=batch["qi"].to(DEVICE); qt=batch["qt"].to(DEVICE); av=batch["avail"].to(DEVICE)
            if DEVICE.type=="cuda": torch.cuda.synchronize()
            t=time.perf_counter()
            _=model(image,tactile,qi,qt,av)
            if DEVICE.type=="cuda": torch.cuda.synchronize()
            times.append((time.perf_counter()-t)*1000/len(image))
    return float(np.mean(times)), float(np.std(times))

lat_mean,lat_sd=benchmark_model(model,test_loader)
profile=pd.DataFrame([{
    "configuration":"fused",
    "trainable_parameters":parameter_count(model),
    "latency_ms_mean":lat_mean,
    "latency_ms_std":lat_sd,
    "device":str(DEVICE)
}])
display(profile)

# Simple ablation switches can be used for controlled retraining.
ABLATIONS = [
    "full_framework",
    "without_quality_descriptors",
    "without_prototype_alignment",
    "without_cross_attention",
    "without_reliability_fusion",
    "without_distillation",
    "without_calibrated_cell_confidence",
    "without_selective_review",
]
print("Ablation configurations:", ABLATIONS)


In [ ]:

# 18. Reliability / risk plots and result export
# Purpose: create manuscript-ready machine-generated result tables without hard-coded values.

# Evaluation table
eval_df.to_csv("braille_framework_evaluation.csv",index=False)

# Training history
history.to_csv("braille_training_history.csv",index=False)

# Confusion matrix
r=evaluate(model,test_loader,"fused",0)
cm=confusion_matrix(r["y"],r["p"])
plt.figure(figsize=(7,6))
plt.imshow(cm,interpolation="nearest")
plt.title("Fused test confusion matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.tight_layout()
plt.savefig("figure_confusion_matrix.png",dpi=300,bbox_inches="tight")
plt.show()

# Accuracy by degradation / availability
pivot=eval_df.pivot(index="severity",columns="availability",values="accuracy")
pivot.plot(marker="o",figsize=(7,4))
plt.ylim(0,1)
plt.ylabel("Accuracy")
plt.xlabel("Degradation severity")
plt.title("Availability × degradation robustness")
plt.grid(alpha=.25)
plt.tight_layout()
plt.savefig("figure_robustness.png",dpi=300,bbox_inches="tight")
plt.show()

print("Saved:")
for f in ["braille_framework_evaluation.csv","braille_training_history.csv",
          "figure_confusion_matrix.png","figure_robustness.png"]:
    print(" -",os.path.abspath(f))



## Real-data configuration notes

For the actual experiment, switch:

```python
CFG.demo_mode = False
```

and set:

```python
CFG.image_root = "/path/to/Multi_Braille_Characters"
CFG.tactile_root = "/path/to/Tactile_Braille_Letters_V2"
CFG.label_map_csv = "/path/to/braille_label_map.csv"
```

The real-data loader intentionally does **not** invent a mapping for unknown dataset labels. Folder names must resolve to a canonical six-dot code, or a dataset-specific label-map parser should be added before training.

The manuscript specifies:
- Multi Braille Characters Dataset: visual images with 105 classes.
- Tactile Braille Letters Dataset V2: 12-taxel event-encoded recordings at 100 Hz.
- Group-aware 70/15/15 partitioning.
- Class-conditional pairing within the same partition.
- Controlled image degradation: blur, low-light/gamma, noise and rotation.
- Controlled tactile degradation: noise, taxel dropout and baseline drift.
- Visual sequence decoding for multi-cell images.
- Deterministic Grade-1 accessible note construction.
- Grad-CAM and integrated gradients for explainability.

These requirements are reflected in the implementation structure above. fileciteturn1file0L39-L51



## What is implemented vs. what must be supplied for the final paper experiment

### Implemented
1. Reproducible configuration and seed control.
2. Canonical six-dot Braille representation.
3. Group-aware train/validation/test partitioning.
4. Three visual and three tactile preprocessing routes.
5. Quality descriptors.
6. Utility prediction and cost-aware route selection.
7. Controlled degradation.
8. BrailleMamba-Lite-style visual encoder.
9. TactileShuffle-Lite-style temporal encoder.
10. BCAN-style cross-modal context and reliability fusion.
11. Shared/private feature decomposition.
12. Prototype-aware training hooks.
13. Missing-modality evaluation.
14. Cell recognition.
15. BRNEN-CG-style confidence/review mechanism.
16. Deterministic note/key-point/key-term generation.
17. Grad-CAM.
18. Tactile integrated gradients.
19. Deletion faithfulness and Hamming error analysis.
20. Evaluation/export utilities.

### Must be completed with the actual datasets before claiming manuscript results
The supplied manuscript explicitly describes cross-fitted reference recognizers, bootstrap-based empirical risk verification, independent policy subsets, repeated group-level partitions and pairing-seed variability. Those should be run on the real datasets rather than represented by synthetic/demo data. The manuscript also states that the reported hardware/software values are planning values and must be replaced with the actual environment before submission. fileciteturn1file1L61-L78

The note-taking layer is deterministic/extractive rather than generative, and the manuscript identifies several limitations including class-conditional rather than physically co-recorded pairing, single-cell tactile data, simulated note-taking, oracle review correction, Grade-1 scope, and synthetic degradation. fileciteturn1file8L749-L774
